# Oral Cancer Screening Using Vision Transformers with Explainable AI
### Mid-Semester Research & Experimentation Notebook

**Project Area:** Healthcare / Oral Oncology Decision Support  
**Target Risk Classes:**
1. `Normal`
2. `Low Risk of Malignant Transformation`
3. `High Risk of Malignant Transformation`

> **Academic & Medical Disclaimer:**  
> This system provides AI-assisted preliminary screening and is not a medical diagnosis. Professional medical evaluation is required for clinical diagnosis.

In [ ]:
# 1. Environment & Setup
import os
import sys
sys.path.append("..")

import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Dataset Verification & Sample Generation
from ml.dataset import generate_sample_dataset, load_dataset_from_directory

sample_dir = generate_sample_dataset("../data/sample", samples_per_class=15)
train_s, val_s, test_s = load_dataset_from_directory(sample_dir)
print(f"Samples: {len(train_s)} train, {len(val_s)} val, {len(test_s)} test")

In [ ]:
# 3. Vision Transformer (ViT-B/16) Architecture Inspection
from ml.vit_model import build_vit_model, CLASS_NAMES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_vit_model(device=device, pretrained=True)

dummy_input = torch.randn(2, 3, 224, 224).to(device)
logits = model(dummy_input)
print("Output Logits Shape:", logits.shape)
print("Classes:", CLASS_NAMES)

In [ ]:
# 4. Monte Carlo Dropout Uncertainty Sampling
from ml.uncertainty import MonteCarloDropoutEstimator

mc = MonteCarloDropoutEstimator(model, num_passes=15)
res = mc.estimate(dummy_input[:1], device=device)
print("Predicted Risk:", res["predicted_class"])
print("Confidence:", res["confidence_pct"])
print("Uncertainty Score:", res["uncertainty_score"], "(", res["uncertainty_level"], ")")
print("Class Probabilities:", res["probabilities"])

In [ ]:
# 5. Explainable AI: ViT-Compatible Grad-CAM
from ml.explainability import generate_explanation_suite

sample_img = Image.open(test_s[0][0]).convert("RGB")
xai_res = generate_explanation_suite(model, dummy_input[:1], sample_img, device=device)
print("Primary Visual Region:", xai_res["primary_region"])
print("XAI Caption:", xai_res["caption"])